In [15]:
"""
Cookie Cats A/B Test - 이탈률(0라운드) 분석
질문: 설치 후 한 번도 플레이 안 한 유저 비율이 두 그룹 간 같은가?
데이터 출처: https://www.kaggle.com/datasets/mursideyarkin/mobile-games-ab-testing-cookie-cats

내려받은 날짜 2026.07.31
"""

'\nCookie Cats A/B Test - 이탈률(0라운드) 분석\n질문: 설치 후 한 번도 플레이 안 한 유저 비율이 두 그룹 간 같은가?\n데이터 출처: https://www.kaggle.com/datasets/mursideyarkin/mobile-games-ab-testing-cookie-cats\n\n내려받은 날짜 2026.07.31\n'

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
 
np.random.seed(42)
 
df = pd.read_csv("cookie_cats.csv")

In [ ]:
# ============================================================
# 1. 데이터 점검
# ============================================================
 
print("shape:", df.shape) #기본 확인
print("결측치:\n", df.isna().sum()) #결측치 확인
print("중복 userid 개수:", df["userid"].duplicated().sum()) # 중복 확인 
print("version 고유값:", df["version"].unique()) # version 확인
 
n_a = (df["version"] == "gate_30").sum()
n_b = (df["version"] == "gate_40").sum()
chi2_srm, p_srm = stats.chisquare([n_a, n_b], [(n_a + n_b) / 2] * 2)
print(f"\nSRM 체크: gate_30={n_a}, gate_40={n_b}")
print(f"chi2={chi2_srm:.4f}, p={p_srm:.4f}")

print("""
메모: 중복/배정 불일치 없음. SRM은 p=0.0086으로 완벽한 50:50은 아니지만
실제 차이는 1%p 미만으로 작아 분석을 진행하되 한계에 기록한다.

사용자를 두 그룹에 나누는 프로그램 코드 자체에 버그가 있어서, 
혹은 특정 조건의 사용자가 한쪽 그룹에 더 많이 들어갔을 가능성이 있다.
(예: 특정 시간대 접속자, 특정 기기 사용자 등이 한쪽으로 쏠림)

정제 전후 행 수 동일(90,189행), 제거한 행 없음.
""")

In [2]:
print("=" * 60)
print("2. 질문과 가설")
print("=" * 60)
print("""
질문: 첫 관문 위치가 온보딩 직후 이탈(설치 후 0라운드 플레이)에 영향을 주는가?
 
H0: 이탈 비율(sum_gamerounds == 0)이 gate_30과 gate_40 간 차이가 없다.
H1: 이탈 비율이 gate_30과 gate_40 간 차이가 있다.
 
alpha = 0.05, 양측검정
""")

2. 질문과 가설

질문: 첫 관문 위치가 온보딩 직후 이탈(설치 후 0라운드 플레이)에 영향을 주는가?

H0: 이탈 비율(sum_gamerounds == 0)이 gate_30과 gate_40 간 차이가 없다.
H1: 이탈 비율이 gate_30과 gate_40 간 차이가 있다.

alpha = 0.05, 양측검정



In [4]:
print("=" * 60)
print("3. 검정 고르기")
print("=" * 60)
print("""
나는 카이제곱 독립성 검정을 골랐다. 왜냐하면 이탈 여부가
1. "했다/안 했다"의 이진 비율 지표이고, 
2. 비교할 집단이 gate_30, gate_40 2개뿐.
""")

3. 검정 고르기

나는 카이제곱 독립성 검정을 골랐다. 왜냐하면 이탈 여부가
1. "했다/안 했다"의 이진 비율 지표이고, 
2. 비교할 집단이 gate_30, gate_40 2개뿐.



In [7]:
print("=" * 60)
print("4. 검정 수행")
print("=" * 60)
 
df["churned"] = df["sum_gamerounds"] == 0
ct = pd.crosstab(df["version"], df["churned"])
chi2, p, dof, expected = stats.chi2_contingency(ct)
 
n_a = ct.loc["gate_30"].sum()
n_b = ct.loc["gate_40"].sum()
p_a = ct.loc["gate_30", True] / n_a
p_b = ct.loc["gate_40", True] / n_b
 
diff = p_a - p_b                    # 절대 차이 (gate_30 - gate_40)
rel = diff / p_b * 100              # 상대 차이(%)
 
se = np.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
ci_low, ci_high = diff - 1.96 * se, diff + 1.96 * se # 신뢰구간 
 
n = ct.values.sum()
cramers_v = np.sqrt(chi2 / n)
 
print(f"gate_30 이탈률: {p_a:.4f} ({ct.loc['gate_30', True]}/{n_a})")
print(f"gate_40 이탈률: {p_b:.4f} ({ct.loc['gate_40', True]}/{n_b})")
print(f"절대 차이: {diff*100:+.2f}%p")
print(f"상대 차이: {rel:+.2f}%")
print(f"95% CI(차이, 신뢰구간): [{ci_low*100:.2f}%p, {ci_high*100:.2f}%p]") #신뢰구간
print(f"chi2({int(dof)}) = {chi2:.2f}, p = {p:.4f}")
print(f"Cramér's V = {cramers_v:.4f}")
 

4. 검정 수행
gate_30 이탈률: 0.0433 (1937/44700)
gate_40 이탈률: 0.0452 (2057/45489)
절대 차이: -0.19%p
상대 차이: -4.17%
95% CI(차이, 신뢰구간): [-0.46%p, 0.08%p]
chi2(1) = 1.85, p = 0.1736
Cramér's V = 0.0045


In [ ]:
"""
5. 한 걸음 더 - A. 이 실험은 효과를 잡을 '힘'이 있었나
retention_1 기준으로 검정력/필요 표본수/MDE를 계산한다.
"""

In [ ]:
!pip install statsmodels

In [11]:
import numpy as np
import pandas as pd
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
 
df = pd.read_csv("cookie_cats.csv")
 
alpha = 0.05
power_target = 0.80
 
# 기준(baseline) 전환율: gate_40(대조군 성격)의 실제 retention_1 비율 사용
p_baseline = df.loc[df["version"] == "gate_40", "retention_1"].mean()
n_per_group_actual = (df["version"] == "gate_40").sum()  # 실제 그룹당 표본 수(참고용)
 
print("=" * 60)
print("A-1. 80% 검정력으로 +1%p 차이를 잡으려면 그룹당 몇 명 필요한가")
print("=" * 60)
 
p_target = p_baseline + 0.01  # +1%p 목표
effect_size = proportion_effectsize(p_target, p_baseline)  # Cohen's h
 
analysis = NormalIndPower()
n_required = analysis.solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power_target,
    ratio=1.0,
    alternative="two-sided",
)
 
print(f"기준 retention_1 (gate_40): {p_baseline:.4f} ({p_baseline*100:.2f}%)")
print(f"목표 retention_1 (+1%p): {p_target:.4f} ({p_target*100:.2f}%)")
print(f"Cohen's h: {effect_size:.4f}")
print(f"필요한 그룹당 표본 수 (80% 검정력, alpha=0.05): {n_required:.0f}명")
print(f"(참고) 실제 gate_40 표본 수: {n_per_group_actual}명")
 
print()
print("=" * 60)
print("A-2. 지금 표본으로 잡을 수 있는 최소 효과(MDE)")
print("=" * 60)
 
# 현재 그룹당 실제 표본 수를 그대로 사용해 effect_size를 역산
mde_effect_size = analysis.solve_power(
    nobs1=n_per_group_actual,
    alpha=alpha,
    power=power_target,
    ratio=1.0,
    alternative="two-sided",
)
 
# Cohen's h -> 비율 차이(%p)로 환산: h가 작을 때 근사적으로
# h ≈ 2*(asin(sqrt(p2)) - asin(sqrt(p1))) 이므로, p_baseline 근처에서
# 수치적으로 p2를 찾아 실제 %p 차이를 구한다.
from scipy.optimize import brentq
 
 
def h_given_p2(p2, p1, h_target):
    return proportion_effectsize(p2, p1) - h_target
 
 
p2_high = brentq(h_given_p2, p_baseline, 0.999, args=(p_baseline, mde_effect_size))
mde_pct_point = (p2_high - p_baseline) * 100
 
print(f"현재 그룹당 표본 수: {n_per_group_actual}명")
print(f"검출 가능한 최소 Cohen's h: {mde_effect_size:.4f}")
print(f"이를 %p로 환산한 MDE: 약 {mde_pct_point:.2f}%p")
 
print()
print("=" * 60)
print("결론")
print("=" * 60)
print(f"""
- 80% 검정력으로 retention_1의 +1%p 차이를 확실히 잡으려면 그룹당 약
  {n_required:,.0f}명이 필요하다. 실제 실험은 그룹당 약 {n_per_group_actual:,}명이었으므로
  표본 수는 {'충분했다' if n_per_group_actual >= n_required else '부족했다'}.
 
- 우리 표본(그룹당 약 {n_per_group_actual:,}명)으로는 최소 약 {mde_pct_point:.2f}%p
  이상의 실제 차이만 80% 확률로 검출할 수 있었다. 즉, retention_1에서 관찰된
  실제 차이(+0.59%p)는 이 표본이 안정적으로 잡아낼 수 있는 최소 효과 크기보다
  {'작아서, 놓쳤을 가능성이 있다' if 0.59 < mde_pct_point else '커서, 검정에서 유의하게 나왔어야 정상이다'}.
 
- 따라서 retention_1에서 p=0.076으로 유의하지 않게 나온 것은 "차이가 없다"는
  확실한 증거라기보다, 이 정도로 작은 차이는 지금 표본 크기로는 안정적으로
  검출하기 어려웠을 가능성을 보여준다.
""")

A-1. 80% 검정력으로 +1%p 차이를 잡으려면 그룹당 몇 명 필요한가
기준 retention_1 (gate_40): 0.4423 (44.23%)
목표 retention_1 (+1%p): 0.4523 (45.23%)
Cohen's h: 0.0201
필요한 그룹당 표본 수 (80% 검정력, alpha=0.05): 38807명
(참고) 실제 gate_40 표본 수: 45489명

A-2. 지금 표본으로 잡을 수 있는 최소 효과(MDE)
현재 그룹당 표본 수: 45489명
검출 가능한 최소 Cohen's h: 0.0186
이를 %p로 환산한 MDE: 약 0.92%p

결론

- 80% 검정력으로 retention_1의 +1%p 차이를 확실히 잡으려면 그룹당 약
  38,807명이 필요하다. 실제 실험은 그룹당 약 45,489명이었으므로
  표본 수는 충분했다.

- 우리 표본(그룹당 약 45,489명)으로는 최소 약 0.92%p
  이상의 실제 차이만 80% 확률로 검출할 수 있었다. 즉, retention_1에서 관찰된
  실제 차이(+0.59%p)는 이 표본이 안정적으로 잡아낼 수 있는 최소 효과 크기보다
  작아서, 놓쳤을 가능성이 있다.

- 따라서 retention_1에서 p=0.076으로 유의하지 않게 나온 것은 "차이가 없다"는
  확실한 증거라기보다, 이 정도로 작은 차이는 지금 표본 크기로는 안정적으로
  검출하기 어려웠을 가능성을 보여준다.



In [13]:
print("\n" + "=" * 60)
print("6. 의사결정 및 한계")
print("=" * 60)
print(f"""
1. 질문과 가설: 관문 위치가 온보딩 초반 이탈(0라운드)에 영향을 주는지 확인했다.
 
2. 데이터 위생 점검: 중복/배정 불일치 없음. SRM이 통계적으로는 유의(p=0.0086)했으나
   차이 크기가 작아(<1%p) 분석에 큰 영향은 없을 것으로 판단.
 
3. 검정 선택 근거: 이진 비율 x 2집단이므로 카이제곱 독립성 검정을 사용했다.
 
4. 결과: gate_30 이탈률 {p_a*100:.2f}%, gate_40 이탈률 {p_b*100:.2f}%,
   절대 차이 {diff*100:+.2f}%p, p={p:.3f} → 유의수준 0.05 기준으로 {'유의함' if p < 0.05 else '유의하지 않음'}.
   Cramér's V = {cramers_v:.4f}로 효과 크기는 매우 작다.
 
5. MDE: 그룹당 {n_b:,}명 표본으로는 80% 검정력 기준 최소 약 {mde_pct_point:.2f}%p 이상의
   차이만 확실히 검출할 수 있었다. 실제 관찰된 차이({abs(diff)*100:.2f}%p)는 이 MDE보다
   {'작아서, 실제로 차이가 있었더라도 이 표본으로는 놓쳤을 가능성이 있다' if abs(diff)*100 < mde_pct_point else '커서, 표본 크기 문제로 놓쳤을 가능성은 낮다'}.
 
6. 의사결정: 이탈률 차이는 통계적으로 {'유의하지만' if p < 0.05 else '유의하지 않고'} 효과 크기가
   극히 작고, MDE 분석 결과 이 정도로 작은 차이는 지금 표본으로 안정적으로
   검출하기 어려운 수준이었다. 관문 위치가 온보딩 초반 이탈에 실질적인
   영향을 준다고 단정하기 어렵다.
   → 이 지표만으로는 출시/보류를 결정할 근거가 부족함 (다른 지표와 함께 판단 필요). 보류!
 
7. 한계:
   - p >= 0.05인 경우, 이는 "차이가 없다는 증명"이 아니라 "이 표본 크기로는
     차이를 확인하지 못했다"는 의미다. 위 5번의 MDE가 그 한계를 구체적인
     숫자로 보여준다.
   - "0라운드 = 이탈"이라는 정의는 편의적 기준이며, 실제로는 설치 직후
     오류/접속 문제로 0라운드가 된 경우와 진짜 흥미를 잃은 경우를 구분하지 못한다.
   - SRM 이슈(1번 참고)로 인해 두 그룹이 완벽히 무작위로 나뉘지 않았을
     가능성이 있어, 인과 해석에는 약한 유보가 필요하다.
""")


6. 의사결정 및 한계

1. 질문과 가설: 관문 위치가 온보딩 초반 이탈(0라운드)에 영향을 주는지 확인했다.

2. 데이터 위생 점검: 중복/배정 불일치 없음. SRM이 통계적으로는 유의(p=0.0086)했으나
   차이 크기가 작아(<1%p) 분석에 큰 영향은 없을 것으로 판단.

3. 검정 선택 근거: 이진 비율 x 2집단이므로 카이제곱 독립성 검정을 사용했다.

4. 결과: gate_30 이탈률 4.33%, gate_40 이탈률 4.52%,
   절대 차이 -0.19%p, p=0.174 → 유의수준 0.05 기준으로 유의하지 않음.
   Cramér's V = 0.0045로 효과 크기는 매우 작다.

5. MDE: 그룹당 45,489명 표본으로는 80% 검정력 기준 최소 약 0.92%p 이상의
   차이만 확실히 검출할 수 있었다. 실제 관찰된 차이(0.19%p)는 이 MDE보다
   작아서, 실제로 차이가 있었더라도 이 표본으로는 놓쳤을 가능성이 있다.

6. 의사결정: 이탈률 차이는 통계적으로 유의하지 않고 효과 크기가
   극히 작고, MDE 분석 결과 이 정도로 작은 차이는 지금 표본으로 안정적으로
   검출하기 어려운 수준이었다. 관문 위치가 온보딩 초반 이탈에 실질적인
   영향을 준다고 단정하기 어렵다.
   → 이 지표만으로는 출시/보류를 결정할 근거가 부족함 (다른 지표와 함께 판단 필요). 보류!

7. 한계:
   - p >= 0.05인 경우, 이는 "차이가 없다는 증명"이 아니라 "이 표본 크기로는
     차이를 확인하지 못했다"는 의미다. 위 5번의 MDE가 그 한계를 구체적인
     숫자로 보여준다.
   - "0라운드 = 이탈"이라는 정의는 편의적 기준이며, 실제로는 설치 직후
     오류/접속 문제로 0라운드가 된 경우와 진짜 흥미를 잃은 경우를 구분하지 못한다.
   - SRM 이슈(1번 참고)로 인해 두 그룹이 완벽히 무작위로 나뉘지 않았을
     가능성이 있어, 인과 해석에는 약한 유보가 필요하다.

